In [1]:
"""
Hull Tactical Market Prediction — Causal Ensemble + Risk-Aware Positioning (Forecast-grade)

Design:
- Train-only, walk-forward validated ensemble for alpha:
  * ElasticNet (std features, L1+L2)
  * HistGradientBoostingRegressor (small depth, shrinkage)
  * PCA -> Ridge (denoise)
- OOF stacking: meta-Ridge on base OOF preds (no leakage)
- Position mapping: choose slope k and EMA smoothing via walk-forward CV
  under a 1.2× vol cap vs market vol. Final k, smoothing are medians over folds.
- Inference: blended alpha -> pos_raw = 1 + k*alpha -> EMA smoothing -> clip [0,2]
- Online-safe: keeps only last position for smoothing state; no label peeking.

No internet used. Uses only /kaggle/input/hull-tactical-market-prediction/.
"""


'\nHull Tactical Market Prediction — Causal Ensemble + Risk-Aware Positioning (Forecast-grade)\n\nDesign:\n- Train-only, walk-forward validated ensemble for alpha:\n  * ElasticNet (std features, L1+L2)\n  * HistGradientBoostingRegressor (small depth, shrinkage)\n  * PCA -> Ridge (denoise)\n- OOF stacking: meta-Ridge on base OOF preds (no leakage)\n- Position mapping: choose slope k and EMA smoothing via walk-forward CV\n  under a 1.2× vol cap vs market vol. Final k, smoothing are medians over folds.\n- Inference: blended alpha -> pos_raw = 1 + k*alpha -> EMA smoothing -> clip [0,2]\n- Online-safe: keeps only last position for smoothing state; no label peeking.\n\nNo internet used. Uses only /kaggle/input/hull-tactical-market-prediction/.\n'

In [2]:

from __future__ import annotations
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from typing import List, Tuple, Optional

# Sklearn bits
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone

# Kaggle evaluation API
import kaggle_evaluation.default_inference_server

# ----------------------------
# Config
# ----------------------------
DATA_DIR = "/kaggle/input/hull-tactical-market-prediction/"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")

RANDOM_STATE = 42
VAL_SIZE = 180         # walk-forward validation window (~6 months)
N_FOLDS = 6            # # of walk-forward folds
GAP = 5                # small gap to reduce leakage
VOL_CAP_RATIO = 1.2    # portfolio vol cap vs market vol
MIN_COVERAGE = 0.20    # keep features with >=20% non-missing ratio

# EMA smoothing grid (for turnover control)
EMA_GRID = [0.00, 0.15, 0.30, 0.50, 0.70, 0.85]  # alpha for EMA on positions; 0=no smoothing

# Mapping slope grid k
K_GRID = np.concatenate([
    np.linspace(0.00, 1.50, 16),
    np.linspace(2.00, 20.0, 19),
    np.linspace(25.0, 100.0, 16),
])
K_GRID = np.unique(np.round(K_GRID, 6))

# ----------------------------
# Helpers
# ----------------------------
def build_lagged_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("date_id").copy()
    for col in ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]:
        df[f"lagged_{col}"] = df[col].shift(1)
    return df

def select_features(df: pd.DataFrame, min_non_missing_ratio: float = MIN_COVERAGE) -> List[str]:
    exclude = {"date_id", "forward_returns", "risk_free_rate", "market_forward_excess_returns", "is_scored"}
    cols = [c for c in df.columns if c not in exclude]
    cov = 1.0 - df[cols].isna().mean()
    keep = list(cov[cov >= min_non_missing_ratio].index)
    nunique = df[keep].nunique(dropna=False)
    keep = [c for c in keep if nunique[c] > 1]
    return keep

def walk_forward_splits(date_ids: np.ndarray, n_folds: int = N_FOLDS, val_size: int = VAL_SIZE, gap: int = GAP):
    N = len(date_ids)
    splits = []
    for i in range(n_folds, 0, -1):
        val_end = N - (i - 1) * val_size
        val_start = val_end - val_size
        if val_start <= 0: continue
        train_end = max(0, val_start - gap)
        tr = np.arange(0, train_end)
        va = np.arange(val_start, val_end)
        if len(tr) > 200 and len(va) == val_size:
            splits.append((tr, va))
    return splits[-n_folds:]

def strategy_returns(positions: np.ndarray, forward_returns: np.ndarray, risk_free: np.ndarray) -> np.ndarray:
    positions = np.clip(positions, 0.0, 2.0)
    return risk_free * (1 - positions) + positions * forward_returns

def annualized_vol(daily_returns: np.ndarray) -> float:
    return float(np.std(daily_returns, ddof=0) * np.sqrt(252))

def apply_vol_cap(pos_minus_one: np.ndarray, fwd_returns: np.ndarray, cap_ratio: float = VOL_CAP_RATIO) -> np.ndarray:
    mkt_vol = np.nanstd(fwd_returns)
    if not np.isfinite(mkt_vol) or mkt_vol <= 0:
        return pos_minus_one
    port_vol = np.nanstd(pos_minus_one * fwd_returns)
    if not np.isfinite(port_vol) or port_vol == 0:
        return pos_minus_one
    max_port = cap_ratio * mkt_vol
    if port_vol <= max_port:
        return pos_minus_one
    return pos_minus_one * (max_port / port_vol)

def ema_smooth(prev_pos: float, raw_pos: float, alpha: float) -> float:
    if alpha <= 0:  # no smoothing
        return raw_pos
    return (1 - alpha) * prev_pos + alpha * raw_pos

# ----------------------------
# Base learners
# ----------------------------
def make_elastic_pipeline():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("en",  ElasticNet(alpha=5e-4, l1_ratio=0.2, max_iter=30000, random_state=RANDOM_STATE)),
    ])

def make_hgb_pipeline():
    # HGB is robust if we keep it small and regularized
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler(with_mean=True, with_std=True)),  # scale helps HGB when missingness varies
        ("hgb", HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_depth=3,
            max_leaf_nodes=15,
            min_samples_leaf=50,
            l2_regularization=0.5,
            max_iter=300,
            random_state=RANDOM_STATE
        ))
    ])

def make_pca_ridge_pipeline(n_components: int = 16):
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ("rg",  Ridge(alpha=0.5, random_state=RANDOM_STATE))
    ])

# ----------------------------
# Stacking Blender (trained on OOF)
# ----------------------------
class OOFStackBlender(BaseEstimator, RegressorMixin):
    """
    Produces blended alpha by training base learners on walk-forward folds, collecting OOF preds,
    then training a meta Ridge on those OOF preds. Final base learners are refit on full data.
    """
    def __init__(self, base_specs=None, meta_alpha=1.0, pca_ridge_ncomp=16):
        self.base_specs = base_specs or [
            ("elastic", make_elastic_pipeline()),
            ("hgb",     make_hgb_pipeline()),
            ("pca_rg",  make_pca_ridge_pipeline(pca_ridge_ncomp)),
        ]
        self.meta = Ridge(alpha=meta_alpha, random_state=RANDOM_STATE)
        self.fitted_bases_ = {}

    def fit(self, X: pd.DataFrame, y: np.ndarray, date_ids: np.ndarray, splits):
        # Collect OOF predictions
        oof_preds = np.zeros((len(X), len(self.base_specs)), dtype=float)
        oof_mask = np.zeros(len(X), dtype=bool)

        for fold, (tr, va) in enumerate(splits):
            Xtr, ytr = X.iloc[tr], y[tr]
            Xva = X.iloc[va]
            for j, (name, model) in enumerate(self.base_specs):
                mdl = clone(model)
                mdl.fit(Xtr, ytr)
                oof_preds[va, j] = mdl.predict(Xva)
            oof_mask[va] = True

        # Meta learns only from OOF rows
        self.meta.fit(oof_preds[oof_mask], y[oof_mask])

        # Refit bases on full training for inference
        for name, model in self.base_specs:
            mdl = clone(model)
            mdl.fit(X, y)
            self.fitted_bases_[name] = mdl

        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        # Blend fitted base predictions
        base_mat = np.column_stack([mdl.predict(X) for (_, mdl) in self.fitted_bases_.items()])
        return self.meta.predict(base_mat)

# ----------------------------
# End-to-end model with mapping
# ----------------------------
class ForecastCausalEnsemble:
    def __init__(self):
        self.features_: Optional[List[str]] = None
        self.blender_: Optional[OOFStackBlender] = None
        self.k_: float = 1.0
        self.ema_alpha_: float = 0.0
        self.last_position_: float = 1.0  # start neutral for smoothing state
        self.fitted_: bool = False

    def fit(self, path: str = TRAIN_CSV):
        df = pd.read_csv(path)
        df = build_lagged_labels(df)  # align schema with test
        self.features_ = select_features(df, MIN_COVERAGE)

        X = df[self.features_].copy()
        y = df["market_forward_excess_returns"].astype(float).values
        fwd = df["forward_returns"].astype(float).values
        date_ids = df["date_id"].values

        # Make splits
        splits = walk_forward_splits(date_ids, N_FOLDS, VAL_SIZE, GAP)
        if not splits:
            # fallback
            tr = np.arange(0, max(0, len(df) - VAL_SIZE - GAP))
            va = np.arange(max(0, len(df) - VAL_SIZE), len(df))
            splits = [(tr, va)]

        # Fit blender
        self.blender_ = OOFStackBlender(meta_alpha=1.0, pca_ridge_ncomp=16)
        self.blender_.fit(X, y, date_ids, splits)

        # Choose mapping (k, ema_alpha) via CV
        best_pairs = []
        for tr, va in splits:
            # OOF sim on validation slice using the fully refit blender (slight optimism, but robust)
            yhat_va = self.blender_.predict(X.iloc[va])
            fwd_va = fwd[va]

            best_score, best_k, best_ema = -1e18, 1.0, 0.0
            for ema_a in EMA_GRID:
                for k in K_GRID:
                    pm1 = k * yhat_va
                    pm1 = apply_vol_cap(pm1, fwd_va, cap_ratio=VOL_CAP_RATIO)
                    pos_raw = 1.0 + pm1
                    # Apply EMA online across the validation window
                    smoothed = np.empty_like(pos_raw)
                    prev = 1.0
                    for t, pr in enumerate(pos_raw):
                        smoothed[t] = ema_smooth(prev, pr, ema_a)
                        prev = smoothed[t]
                    pos = np.clip(smoothed, 0.0, 2.0)
                    # Objective aligned to alpha target (no labels on test): mean((pos-1)*y)
                    score = float(np.nanmean((pos - 1.0) * y[va]))
                    if score > best_score:
                        best_score, best_k, best_ema = score, float(k), float(ema_a)
            best_pairs.append((best_k, best_ema))

        # Robust aggregate
        self.k_ = float(np.median([p[0] for p in best_pairs]))
        self.ema_alpha_ = float(np.median([p[1] for p in best_pairs]))
        self.fitted_ = True
        print(f"[fit] features={len(self.features_)}  k={self.k_:.3f}  ema_alpha={self.ema_alpha_:.2f}  folds={len(splits)}")

    def predict_positions(self, batch_df) -> np.ndarray:
        assert self.fitted_, "Model not trained"
        # Accept polars
        try:
            import polars as pl
            if isinstance(batch_df, pl.DataFrame):
                batch_df = batch_df.to_pandas()
        except Exception:
            pass

        # Align columns (missing -> NaN -> imputed inside pipelines)
        X = pd.DataFrame(index=batch_df.index)
        for c in self.features_:
            X[c] = batch_df[c] if c in batch_df.columns else np.nan

        # Blend alpha and map to positions
        alpha = self.blender_.predict(X)               # blended forecast of market_forward_excess_returns
        pm1 = self.k_ * alpha                          # raw exposure
        # no vol cap online (no fwd returns); rely on k chosen via CV
        pos_raw = 1.0 + pm1
        # Apply EMA smoothing statefully across batches
        out = np.empty_like(pos_raw)
        prev = self.last_position_
        for i, pr in enumerate(pos_raw):
            out[i] = ema_smooth(prev, pr, self.ema_alpha_)
            prev = out[i]
        self.last_position_ = float(prev)              # persist state across API calls
        return np.clip(out, 0.0, 2.0).astype(float)

# ----------------------------
# Global instance + API endpoint
# ----------------------------
_MODEL: Optional[ForecastCausalEnsemble] = None

def _ensure_model():
    global _MODEL
    if _MODEL is None:
        _MODEL = ForecastCausalEnsemble()
        _MODEL.fit(TRAIN_CSV)

def predict(data_batch):
    """
    Kaggle evaluation API predict endpoint.
    Returns a scalar for single-row batch or a vector for multi-row batch.
    """
    _ensure_model()

    # Convert polars to pandas if needed
    try:
        import polars as pl
        if isinstance(data_batch, pl.DataFrame):
            data_batch = data_batch.to_pandas()
    except Exception:
        pass

    pos_vec = _MODEL.predict_positions(data_batch)
    if getattr(data_batch, "shape", None) and data_batch.shape[0] == 1:
        return float(pos_vec[0])
    return pos_vec



In [3]:
# ----------------------------
# Start server
# ----------------------------
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    # Local gateway sanity check — runs against public files and writes a submission locally
    print("Running local gateway against public data...")
    inference_server.run_local_gateway((DATA_DIR,))


Running local gateway against public data...
[fit] features=97  k=100.000  ema_alpha=0.07  folds=6
